# 3D Gaussian Splatting Playground - Quick Start

This notebook walks you through the full pipeline: from raw images to a trained 3D Gaussian Splatting model you can view in your browser.

---

このノートブックでは、生画像から 3D Gaussian Splatting モデルの学習・ビューアでの表示までの一連のパイプラインを体験できます。

In [ ]:
# !pip install -e "..[dev]"

## 1. Prepare Images

In [ ]:
from gs_sim2real.common.download import download_sample_images
from gs_sim2real.common.config import get_project_root
from pathlib import Path

data_dir = get_project_root() / "data"
image_dir = data_dir / "sample_images"

# Download sample images for demo
download_sample_images(str(image_dir), num_images=20)
print(f"Images saved to: {image_dir}")
print(f"Number of images: {len(list(image_dir.glob('*.jpg')) + list(image_dir.glob('*.png')))}")

## 2. Extract Frames from Video (Optional)

In [ ]:
from gs_sim2real.preprocess.extract_frames import extract_frames

# If you have a video file:
# frames = extract_frames("path/to/video.mp4", str(data_dir / "frames"), fps=2, max_frames=100)
# print(f"Extracted {len(frames)} frames")

## 3. COLMAP Preprocessing

In [ ]:
from gs_sim2real.preprocess.colmap import COLMAPProcessor

colmap_output = data_dir / "colmap_output"

try:
    processor = COLMAPProcessor()
    sparse_dir = processor.run_sfm(
        image_dir=str(image_dir),
        output_dir=str(colmap_output),
        use_gpu=True
    )
    print(f"COLMAP sparse model at: {sparse_dir}")
except FileNotFoundError:
    print("COLMAP is not installed. Install it:")
    print("  Ubuntu: sudo apt install colmap")
    print("  macOS: brew install colmap")
    print("  Or download from: https://colmap.github.io/")

## 4. Train 3D Gaussian Splatting

In [ ]:
from gs_sim2real.train.gsplat_trainer import GsplatTrainer

trainer = GsplatTrainer()

output_dir = get_project_root() / "outputs" / "gs_model"

try:
    trainer.train(
        data_dir=str(colmap_output),
        output_dir=str(output_dir),
        num_iterations=1000  # Short for demo, use 30000 for quality
    )
    print(f"Model saved to: {output_dir}")
except Exception as e:
    print(f"Training error: {e}")
    print("Make sure gsplat is installed: pip install gsplat")

## 4b. Alternative: Train with nerfstudio

In [ ]:
# Alternative: use nerfstudio's splatfacto
# from gs_sim2real.train.nerfstudio_trainer import NerfstudioTrainer
# 
# ns_trainer = NerfstudioTrainer()
# ns_trainer.train(
#     data_dir=str(colmap_output),
#     output_dir=str(output_dir / "nerfstudio"),
#     method="splatfacto",
#     num_iterations=1000
# )

## 5. Visualize Results

In [ ]:
from gs_sim2real.viewer.web_viewer import GaussianViewer

ply_path = output_dir / "point_cloud.ply"

if ply_path.exists():
    viewer = GaussianViewer(port=8080)
    print("Starting web viewer...")
    print("Open http://localhost:8080 in your browser")
    # viewer.view_ply(str(ply_path))  # Uncomment to launch (blocks)
else:
    print(f"No model found at {ply_path}")
    print("Run the training step first.")

## 6. Full Pipeline (One Command)

In [ ]:
# You can also run the full pipeline from command line:
# !gs-sim2real run --images data/sample_images/ --output outputs/gs_model/ --iterations 1000

## Tips

### Training
- Use **at least 20-50 images** with good coverage around the object/scene.
- For production quality, train for **30,000 iterations** (takes ~15-30 min on a modern GPU).
- GPU with **8 GB+ VRAM** is recommended. Use `use_gpu=False` for CPU-only COLMAP if needed.

### Image Capture
- Capture images from **many different angles** with sufficient overlap.
- Avoid motion blur and ensure consistent lighting.
- For video input, extract at **2-3 fps** to avoid redundant frames.

### References
- [3D Gaussian Splatting (original paper)](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/)
- [gsplat library](https://docs.gsplat.studio/)
- [nerfstudio](https://docs.nerf.studio/)
- [COLMAP](https://colmap.github.io/)